# QDMpy Tutorial: ODMR Data Analysis

This tutorial demonstrates how to use QDMpy for analyzing Optically Detected Magnetic Resonance (ODMR) data from Quantum Diamond Microscopy (QDM) experiments.

## Overview

QDMpy provides a complete workflow for ODMR analysis:
1. **Load** data from various formats (.mat, .csv)
2. **Process** data with built-in processors (binning, normalization, outlier removal)
3. **Fit** spectra using physics-based models
4. **Visualize** results with built-in plotting functions
5. **Export** processed data and results

Let's walk through a complete analysis workflow using real data.

In [ ]:
# Import QDMpy modules
import numpy as np
import matplotlib.pyplot as plt

# Core QDMpy imports
from QDMpy.odmr import ODMRData, ODMR
from QDMpy.odmr.io import MatlabLoader
from QDMpy.odmr.processors import (
    BinningProcessor, 
    NormalizationProcessor, 
    OutlierProcessor,
    FluorescenceCorrectionProcessor
)
from QDMpy.models import ModelRegistry
from QDMpy.guess import guess_n_peaks, guess_model, guess_initial_fit_parameters
from QDMpy.fit import Fit
from QDMpy.measurement import Measurement
from QDMpy.io import get_image, has_csv

print("QDMpy imported successfully!")

In [2]:
# Import QDMpy modules
import numpy as np
import matplotlib.pyplot as plt

# Core QDMpy imports
from QDMpy.odmr import ODMRData, ODMR
from QDMpy.odmr.io import MatlabLoader
from QDMpy.odmr.processors import (
    BinningProcessor, 
    NormalizationProcessor, 
    OutlierProcessor,
    FluorescenceCorrectionProcessor
)
from QDMpy.models import ModelRegistry
from QDMpy.guess import guess_n_peaks, guess_model, guess_initial_fit_parameters
from QDMpy.fit import Fit
from QDMpy.measurement import Measurement
from QDMpy.plotting import plot_odmr_spectrum, plot_parameter_map
from QDMpy.io import get_image, has_csv

print("QDMpy imported successfully!")

21:00:03     INFO QDMpy.<module> >> WELCOME TO QDMpy
21:00:03.%f     INFO QDMpy.<module> >> WELCOME TO QDMpy
21:00:03    DEBUG QDMpy.<module> >> QDMpy version 0.1.0a installed at /home/mike/git/QDMpy/src/QDMpy
21:00:03.%f    DEBUG QDMpy.<module> >> QDMpy version 0.1.0a installed at /home/mike/git/QDMpy/src/QDMpy
21:00:03    DEBUG QDMpy.<module> >> QDMpy config file /home/mike/.config/QDMpy/config.ini
21:00:03.%f    DEBUG QDMpy.<module> >> QDMpy config file /home/mike/.config/QDMpy/config.ini
21:00:03     INFO QDMpy.load_config >> Loading config file: /home/mike/.config/QDMpy/config.ini
21:00:03.%f     INFO QDMpy.load_config >> Loading config file: /home/mike/.config/QDMpy/config.ini
0 <class 'ctypes.c_ulong'>
1 <class 'ctypes.c_ulong'>
2 <class 'pygpufit.gpufit.LP_c_float'>
3 <class 'pygpufit.gpufit.LP_c_float'>
4 <class 'ctypes.c_int'>
5 <class 'pygpufit.gpufit.LP_c_float'>
6 <class 'pygpufit.gpufit.LP_c_float'>
7 <class 'pygpufit.gpufit.LP_c_int'>
8 <class 'ctypes.c_float'>
9 <class 

ImportError: cannot import name 'ODMRdata' from 'QDMpy.odmr.data' (/home/mike/git/QDMpy/src/QDMpy/odmr/data.py)

In [ ]:
## 1. Loading ODMR Data

QDMpy can load data from various formats. Let's load sample data from the test directory.

In [ ]:
# Load ODMR data using MatlabLoader
data_folder = "./tests/data"
loader = MatlabLoader(data_folder=data_folder)

# Load the data - returns raw_data, scan_dimensions, frequencies
raw_data, scan_dimensions, frequencies = loader.load()

# Create ODMRData object
odmr_data = ODMRData(
    data=raw_data,
    scan_dimensions=scan_dimensions, 
    frequencies=frequencies
)

print(f"Loaded ODMR data:")
print(f"  Shape: {odmr_data.shape}")
print(f"  Scan dimensions: {odmr_data.scan_dimensions}")
print(f"  Frequency range: {odmr_data.frequencies.min()/1e9:.3f} - {odmr_data.frequencies.max()/1e9:.3f} GHz")
print(f"  Number of pixels: {odmr_data.shape[2]}")
print(f"  Number of frequency points: {odmr_data.shape[3]}")

In [ ]:
## 2. Processing ODMR Data

QDMpy provides a modular processing pipeline. Let's set up and apply common processing steps.

In [ ]:
# Create ODMR manager for processing
odmr = ODMR(odmr_data)

# Add processing steps to the pipeline
print("Setting up processing pipeline:")

# 1. Remove outliers (optional)
odmr.processor_manager.add_processor(OutlierProcessor(threshold=0.02))
print("  ✓ Outlier removal")

# 2. Apply fluorescence correction (optional)
odmr.processor_manager.add_processor(FluorescenceCorrectionProcessor(correction_factor=0.5))
print("  ✓ Fluorescence correction")

# 3. Spatial binning to reduce noise
odmr.processor_manager.add_processor(BinningProcessor(bin_factor=2))
print("  ✓ Spatial binning (2x2)")

# 4. Normalize data
odmr.processor_manager.add_processor(NormalizationProcessor(method='max'))
print("  ✓ Normalization (max)")

# Apply all processing steps
print("\nProcessing data...")
odmr.process_data()

print(f"Processing complete!")
print(f"  Original shape: {odmr.raw_data.shape}")
print(f"  Processed shape: {odmr.processed_data.shape}")
print(f"  Processing metadata: {list(odmr.processed_data.metadata.keys())}")

In [ ]:
# Plot raw spectrum for a sample pixel
pixel_idx = 100
spectrum = odmr.raw_data.data[0, 0, :, pixel_idx]  # First polarity, first range

plt.figure(figsize=(10, 6))
plt.plot(odmr.raw_data.frequencies / 1e9, spectrum, 'b-', linewidth=2)
plt.xlabel('Frequency (GHz)')
plt.ylabel('ODMR Signal')
plt.title(f'Raw ODMR Spectrum - Pixel {pixel_idx}')
plt.grid(True, alpha=0.3)
plt.show()

# Plot processed spectrum for comparison
pixel_idx = 25  # Adjusted for binning
spectrum = odmr.processed_data.data[0, 0, :, pixel_idx]

plt.figure(figsize=(10, 6))
plt.plot(odmr.processed_data.frequencies / 1e9, spectrum, 'r-', linewidth=2)
plt.xlabel('Frequency (GHz)')
plt.ylabel('ODMR Signal (normalized)')
plt.title(f'Processed ODMR Spectrum - Pixel {pixel_idx}')
plt.grid(True, alpha=0.3)
plt.show()

# Plot raw spectrum for a sample pixel using QDMpy's plotting
try:
    plot_odmr_spectrum(
        odmr.raw_data.data, 
        odmr.raw_data.frequencies, 
        pixel_idx=100,
        title="Raw ODMR Spectrum"
    )
except ImportError:
    # Fallback to simple matplotlib if QDMpy plotting not available
    pixel_idx = 100
    spectrum = odmr.raw_data.data[0, 0, :, pixel_idx]  # First polarity, first range
    
    plt.figure(figsize=(10, 6))
    plt.plot(odmr.raw_data.frequencies / 1e9, spectrum, 'b-', linewidth=2)
    plt.xlabel('Frequency (GHz)')
    plt.ylabel('ODMR Signal')
    plt.title(f'Raw ODMR Spectrum - Pixel {pixel_idx}')
    plt.grid(True, alpha=0.3)
    plt.show()

# Plot processed spectrum for comparison
try:
    plot_odmr_spectrum(
        odmr.processed_data.data, 
        odmr.processed_data.frequencies, 
        pixel_idx=25,  # Adjusted for binning
        title="Processed ODMR Spectrum"
    )
except ImportError:
    # Fallback to simple matplotlib
    pixel_idx = 25  # Adjusted for binning
    spectrum = odmr.processed_data.data[0, 0, :, pixel_idx]
    
    plt.figure(figsize=(10, 6))
    plt.plot(odmr.processed_data.frequencies / 1e9, spectrum, 'r-', linewidth=2)
    plt.xlabel('Frequency (GHz)')
    plt.ylabel('ODMR Signal (normalized)')
    plt.title(f'Processed ODMR Spectrum - Pixel {pixel_idx}')
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# Detect number of peaks in the data
print("Analyzing ODMR spectra...")
n_peaks, doubt, peak_indices = guess_n_peaks(odmr.processed_data.data)

print(f"Peak detection results:")
print(f"  Number of peaks detected: {n_peaks}")
print(f"  Confidence: {'Low (uncertain)' if doubt else 'High (confident)'}")

# List available models
print(f"\nAvailable models in QDMpy:")
for name, model_info in ModelRegistry.all().items():
    model_instance = model_info["class"]()
    print(f"  {name}: {model_instance.n_peaks} peaks")

# Automatically select appropriate model
try:
    model = guess_model(n_peaks)
    print(f"\nSelected model: {model.name}")
    print(f"  Number of peaks: {model.n_peaks}")
    print(f"  Parameters: {model.parameters_unique}")
except Exception as e:
    print(f"\nModel selection failed: {e}")
    # Fallback to a default model
    model = ModelRegistry.get("ESRSINGLE")
    print(f"Using fallback model: {model.name}")

# Detect number of peaks in the data
print("Analyzing ODMR spectra...")
n_peaks, doubt, peak_indices = guess_n_peaks(odmr.processed_data.data)

print(f"Peak detection results:")
print(f"  Number of peaks detected: {n_peaks}")
print(f"  Confidence: {'Low (uncertain)' if doubt else 'High (confident)'}")

# List available models
print(f"\nAvailable models in QDMpy:")
for name, model_info in ModelRegistry.all().items():
    model_instance = model_info["class"]()
    print(f"  {name}: {model_instance.n_peaks} peaks")

# Automatically select appropriate model
try:
    model = guess_model(n_peaks)
    print(f"\nSelected model: {model.name}")
    print(f"  Number of peaks: {model.n_peaks}")
    print(f"  Parameters: {model.parameters_unique}")
except Exception as e:
    print(f"\nModel selection failed: {e}")
    # Fallback to a default model
    model = ModelRegistry.get("ESRSINGLE")
    print(f"Using fallback model: {model.name}")

In [ ]:
## 5. Fitting ODMR Data

Now let's fit the processed data using the selected model.

# Create fit object with the selected model
print("Setting up fitting...")
fit_obj = Fit(
    data=odmr.processed_data.data,
    frequencies=odmr.processed_data.frequencies,
    model_name=model.name
)

print(f"Fit configuration:")
print(f"  Model: {fit_obj.model_name}")
print(f"  Parameters: {fit_obj.model_params_unique}")
print(f"  Data shape: {fit_obj._data.shape}")

# Generate initial parameter guesses
print("\nGenerating initial parameter guesses...")
initial_params = guess_initial_fit_parameters(
    data=odmr.processed_data.data,
    freq=odmr.processed_data.frequencies, 
    model=model
)
print(f"Initial parameters shape: {initial_params.shape}")

# Perform the fit
print("\nPerforming fit...")
try:
    fit_obj.fit_odmr()
    print(f"✓ Fit completed successfully!")
    print(f"  Fitted: {fit_obj.fitted}")
    print(f"  Results shape: {fit_obj.parameter.shape}")
except Exception as e:
    print(f"✗ Fit failed: {e}")
    print("This may be due to data format or model compatibility issues.")

In [ ]:
## 6. Visualizing Fit Results

Let's visualize the fitting results for individual pixels.

In [ ]:
# Plot fit results for a sample pixel
if fit_obj.fitted:
    print("Plotting fit results...")
    
    # Select a pixel to examine
    pixel_idx = 10
    polarity_idx = 0
    freq_range_idx = 0
    
    # Get the data and frequencies
    data = fit_obj._data[polarity_idx, freq_range_idx, :, pixel_idx]
    frequencies = fit_obj.frequencies
    
    # Get fit parameters for this pixel
    params = fit_obj.parameter[polarity_idx, freq_range_idx, pixel_idx]
    
    # Generate fitted curve
    fitted_curve = fit_obj.model_func(frequencies, params)
    
    # Create the plot
    plt.figure(figsize=(12, 6))
    plt.plot(frequencies / 1e9, data, 'bo', markersize=4, label='Data', alpha=0.7)
    plt.plot(frequencies / 1e9, fitted_curve, 'r-', linewidth=2, label='Fit')
    
    plt.xlabel('Frequency (GHz)')
    plt.ylabel('ODMR Signal (normalized)')
    plt.title(f'ODMR Fit Results - Pixel {pixel_idx}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Add fit parameters as text
    param_text = "Fit Parameters:\\n"
    for i, param in enumerate(fit_obj.model_params_unique):
        value = params[i]
        if 'center' in param.lower():
            param_text += f"{param}: {value/1e9:.6f} GHz\\n"
        elif 'width' in param.lower():
            param_text += f"{param}: {value/1e6:.3f} MHz\\n"
        else:
            param_text += f"{param}: {value:.6f}\\n"
    
    plt.text(0.02, 0.98, param_text, transform=plt.gca().transAxes, 
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    plt.show()
    
    print(f"Fit quality metrics for pixel {pixel_idx}:")
    try:
        chi_square = fit_obj.chi_square[polarity_idx, freq_range_idx, pixel_idx]
        iterations = fit_obj.iterations[polarity_idx, freq_range_idx, pixel_idx]
        print(f"  Chi-square: {chi_square:.6f}")
        print(f"  Iterations: {iterations}")
    except:
        print("  Quality metrics not available")
        
else:
    print("No fit results available to plot.")

## 7. Creating a Measurement Object

QDMpy's Measurement class combines ODMR data with optical images for comprehensive analysis.

In [ ]:
# Try to load optical images from the data folder
print("Loading optical images...")
height, width = odmr.processed_data.scan_dimensions
light_image = np.random.random((height, width))  # Default dummy image
laser_image = np.random.random((height, width))  # Default dummy image

try:
    import os
    file_list = os.listdir(data_folder)
    
    if has_csv(file_list):
        print("  CSV files detected, attempting to load images...")
        
        # Try to load light image
        try:
            light_candidates = [f for f in file_list if "led" in f.lower()]
            if light_candidates:
                light_image = get_image(data_folder, light_candidates)
                if light_image.shape == (height, width):
                    print("  ✓ Light image loaded successfully")
                else:
                    print(f"  ⚠ Light image shape mismatch: {light_image.shape} vs {(height, width)}")
        except Exception as e:
            print(f"  ⚠ Could not load light image: {e}")
        
        # Try to load laser image  
        try:
            laser_candidates = [f for f in file_list if "laser" in f.lower()]
            if laser_candidates:
                laser_image = get_image(data_folder, laser_candidates)
                if laser_image.shape == (height, width):
                    print("  ✓ Laser image loaded successfully")
                else:
                    print(f"  ⚠ Laser image shape mismatch: {laser_image.shape} vs {(height, width)}")
        except Exception as e:
            print(f"  ⚠ Could not load laser image: {e}")
    else:
        print("  No CSV files found, using dummy images")
        
except Exception as e:
    print(f"  Error accessing data folder: {e}")
    print("  Using dummy images")

# Create measurement object
measurement = Measurement(
    odmr=odmr,
    light_image=light_image,
    laser_image=laser_image,
    output_directory="./output",
    pixel_spacing=4e-6,  # 4 μm pixel spacing
    fit_model=model.name
)

# Add metadata
measurement.metadata.update({
    "experiment_date": "2024-12-07",
    "sample": "Tutorial sample", 
    "processing_steps": ["outlier_removal", "fluorescence_correction", "binning", "normalization"],
    "bin_factor": 2,
    "notes": "QDMpy tutorial analysis"
})

print(f"\\nMeasurement object created:")
print(f"  ODMR data shape: {measurement.odmr.processed_data.shape}")
print(f"  Light image shape: {measurement.light_image.shape}")
print(f"  Laser image shape: {measurement.laser_image.shape}")
print(f"  Pixel spacing: {measurement.pixel_spacing*1e6:.1f} μm")

## 8. Visualizing Results

Let's create a comprehensive visualization of our analysis results.

In [ ]:
# Create a comprehensive visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Plot light image
im1 = axes[0, 0].imshow(measurement.light_image, cmap='gray')
axes[0, 0].set_title('Light Image')
axes[0, 0].axis('off')
plt.colorbar(im1, ax=axes[0, 0], shrink=0.8)

# Plot laser image  
im2 = axes[0, 1].imshow(measurement.laser_image, cmap='hot')
axes[0, 1].set_title('Laser Image')
axes[0, 1].axis('off')
plt.colorbar(im2, ax=axes[0, 1], shrink=0.8)

# Plot ODMR contrast map (if available)
try:
    # Calculate contrast as max - min for each pixel
    contrast_data = odmr.processed_data.data[0, 0]  # First polarity and range
    contrast_map = np.max(contrast_data, axis=1) - np.min(contrast_data, axis=1)
    contrast_map = contrast_map.reshape(height, width)
    
    im3 = axes[0, 2].imshow(contrast_map, cmap='viridis')
    axes[0, 2].set_title('ODMR Contrast Map')
    axes[0, 2].axis('off')
    plt.colorbar(im3, ax=axes[0, 2], shrink=0.8)
except Exception as e:
    axes[0, 2].text(0.5, 0.5, f'Contrast map\\nNot Available\\n{str(e)[:30]}...', 
                   ha='center', va='center', transform=axes[0, 2].transAxes)
    axes[0, 2].set_title('ODMR Contrast Map')
    axes[0, 2].axis('off')

# Plot fit parameter maps (if fitting was successful)
if fit_obj.fitted:
    try:
        # Plot center frequency map
        center_idx = [i for i, p in enumerate(fit_obj.model_params_unique) if 'center' in p.lower()][0]
        center_values = fit_obj.parameter[0, 0, :, center_idx]  # First polarity and range
        center_map = center_values.reshape(height, width)
        
        im4 = axes[1, 0].imshow(center_map / 1e9, cmap='plasma')  # Convert to GHz
        axes[1, 0].set_title('Center Frequency (GHz)')
        axes[1, 0].axis('off')
        plt.colorbar(im4, ax=axes[1, 0], shrink=0.8)
        
        # Plot width map (if available)
        width_params = [i for i, p in enumerate(fit_obj.model_params_unique) if 'width' in p.lower()]
        if width_params:
            width_idx = width_params[0]
            width_values = fit_obj.parameter[0, 0, :, width_idx]
            width_map = width_values.reshape(height, width)
            
            im5 = axes[1, 1].imshow(width_map / 1e6, cmap='plasma')  # Convert to MHz
            axes[1, 1].set_title('Linewidth (MHz)')
            axes[1, 1].axis('off')
            plt.colorbar(im5, ax=axes[1, 1], shrink=0.8)
        else:
            axes[1, 1].text(0.5, 0.5, 'Width parameter\\nnot available', 
                           ha='center', va='center', transform=axes[1, 1].transAxes)
            axes[1, 1].set_title('Linewidth')
            axes[1, 1].axis('off')
        
        # Plot contrast parameter map (if available)
        contrast_params = [i for i, p in enumerate(fit_obj.model_params_unique) if 'contrast' in p.lower()]
        if contrast_params:
            contrast_idx = contrast_params[0]
            contrast_values = fit_obj.parameter[0, 0, :, contrast_idx]
            contrast_fit_map = contrast_values.reshape(height, width)
            
            im6 = axes[1, 2].imshow(contrast_fit_map, cmap='viridis')
            axes[1, 2].set_title('Fitted Contrast')
            axes[1, 2].axis('off')
            plt.colorbar(im6, ax=axes[1, 2], shrink=0.8)
        else:
            axes[1, 2].text(0.5, 0.5, 'Contrast parameter\\nnot available', 
                           ha='center', va='center', transform=axes[1, 2].transAxes)
            axes[1, 2].set_title('Fitted Contrast')
            axes[1, 2].axis('off')
            
    except Exception as e:
        for i in range(3):
            axes[1, i].text(0.5, 0.5, f'Fit parameter maps\\nnot available\\n{str(e)[:20]}...', 
                           ha='center', va='center', transform=axes[1, i].transAxes)
            axes[1, i].axis('off')
else:
    for i in range(3):
        axes[1, i].text(0.5, 0.5, 'Fit parameters\\nnot available\\n(fitting failed)', 
                       ha='center', va='center', transform=axes[1, i].transAxes)
        axes[1, i].axis('off')

plt.suptitle('QDMpy Analysis Results', fontsize=16)
plt.tight_layout()
plt.show()

print("Analysis visualization complete!")

## 9. Summary and Next Steps

This tutorial demonstrated the complete QDMpy workflow:

### What we accomplished:
✅ **Data Loading**: Loaded ODMR data from MATLAB files using `MatlabLoader`  
✅ **Data Processing**: Applied outlier removal, fluorescence correction, binning, and normalization  
✅ **Model Selection**: Automatically detected peaks and selected appropriate fitting model  
✅ **Fitting**: Fitted ODMR spectra to extract physical parameters  
✅ **Visualization**: Created comprehensive plots of results  
✅ **Measurement Object**: Combined ODMR data with optical images for complete analysis  

### Key QDMpy Features Used:
- **`ODMRData`**: Core data container with metadata tracking
- **`ODMR`**: Processing pipeline manager with modular processors
- **`ModelRegistry`**: Automatic model selection based on data characteristics  
- **`Fit`**: GPU-accelerated fitting with constraint management
- **`Measurement`**: Integration of ODMR data with optical imaging
- **Built-in processors**: Outlier removal, fluorescence correction, binning, normalization

### Next Steps:
1. **Explore other models**: Try different spectral models (ESR14N, ESR15N) for multi-peak data
2. **Custom processing**: Create custom processors for specialized data transformations
3. **Advanced fitting**: Set custom constraints and initial parameters for improved fits
4. **Batch analysis**: Process multiple datasets using QDMpy's CLI tools
5. **Export results**: Save processed data and fit results for further analysis

### Getting Help:
- Check the [QDMpy documentation](docs/) for detailed API references
- Run `qdmpy --help` for CLI usage information  
- See `examples/` folder for more analysis scripts
- Look at specialized tutorials for [fitting](tutorial_fitting.ipynb) and [models](tutorial_models.ipynb)

**QDMpy makes ODMR analysis reproducible, efficient, and accessible!**